# VIX Deep Learning Pipeline — Version Améliorée
## Framework institutionnel : Prédiction d'amplitude et de direction du VIX

Ce notebook implémente un pipeline complet combinant :
- Feature engineering avancé (EGARCH, Kalman, HMM, Heston proxies, VRP, Hawkes)
- Architectures deep learning : LSTM, TCN, Transformer, CNN-LSTM, Transformer-LSTM, **N-BEATS**, **Temporal Fusion Transformer**
- Régimes de marché appris (GMM + HMM)
- Ensemble pondéré par performance walk-forward
- SHAP pour l'interprétabilité
- Validation walk-forward institutionnelle avec gap


## 0. Installation et imports

**Pourquoi ces librairies ?**
- `arch` : modèles GARCH/EGARCH pour la variance conditionnelle
- `pykalman` : filtre de Kalman pour extraire l'état latent du VIX
- `hmmlearn` : Hidden Markov Model pour les régimes cachés
- `pytorch-forecasting` : implémente le Temporal Fusion Transformer (TFT), architecture state-of-the-art pour les séries temporelles
- `shap` : SHapley Additive exPlanations — interprétabilité des modèles


In [14]:
import sys
!{sys.executable} -m pip install -q arch pykalman hmmlearn shap pytorch-forecasting pytorch-lightning xlsxwriter

import os, time, random, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.mixture import GaussianMixture
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                              precision_score, recall_score, confusion_matrix)
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from imblearn.combine import SMOTETomek

import matplotlib.pyplot as plt
import seaborn as sns
import shap

from arch import arch_model
from pykalman import KalmanFilter
from hmmlearn import hmm as hmmlib

# ── Reproductibilité totale ──────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")


Device : cpu


## 1. Configuration centrale

**Principe** : toute constante est définie ici une seule fois.
Le split 80/20 chronologique est calculé dynamiquement sur l'index du dataset
pour éviter toute fuite d'information (data leakage).

**Horizons** : 1j, 3j, 5j, 7j — chaque horizon génère une cible indépendante.
**Lookback** : 21 jours de bourse (~1 mois calendaire) en entrée des modèles séquentiels.


In [15]:
CONFIG = {
    'seed':         42,
    'start_date':   '2012-01-01',
    'lookback':     21,       # fenêtre temporelle en entrée (LSTM/TCN/Transformer)
    'horizons':     [1, 3, 5, 7],
    'flat_thr':     0.003,    # jours flat supprimés (|rendement| < 0.3%)
    'top_n_shap':   40,       # features base → SHAP pilote
    'top_n_final':  30,       # features finales après interactions
    'batch_size':   64,
    'epochs':       60,
    'lr':           3e-4,
    'weight_decay': 1e-4,
    'dropout':      0.3,
    # Classes amplitude (quantiles conditionnels calculés sur le train)
    'class_quantiles': [0.25, 0.75],
    'class_labels': ['DOWN_FORT','DOWN_FAIBLE','UP_FAIBLE','UP_FORT'],
}

# Tickers Yahoo Finance (77 séries validées par SHAP dans les runs précédents)
YF_TICKERS = """
^GSPC ^IXIC ^VIX ^VXN ^OVX ^GVZ ^EVZ ^VVIX
^FTSE ^N225 ^HSI ^GDAXI ^STOXX50E
SPY QQQ TLT GLD USO HYG LQD
AAPL AMZN MSFT NVDA INTC QCOM CSCO XOM WMT MCD SBUX
MS COF BLK SCHW CLX CPB GIS BMY CI BDX
LMT NOC GD HON DE ITT
CCI AVB PSA EQIX EXC NEE
TXN LOGI VOD ENB PAYX LUV CMCSA
""".split()

FRED_SERIES = {
    'NFCI':   'NFCI',
    'STLFSI': 'STLFSI4',
    'T10Y2Y': 'T10Y2Y',
    'EFFR':   'EFFR',
    'VXVCLS': 'VXVCLS',  # VIX 3-month
}

TARGET_COL = 'VIX_Amplitude_Class'  # 4 classes
print("Configuration chargée.")


Configuration chargée.


## 2. Chargement des données

**Principe de couverture** : on télécharge tous les tickers puis on filtre
par colonne (≥ 90% de données non-NaN) plutôt que par ligne.
Cela évite de perdre des années entières à cause d'un ticker tardif.

**FRED** : Forward-fill (ffill) pour propager les publications mensuelles
sur le calendrier boursier journalier. La valeur de CPI du 15 janvier
est utilisée jusqu'au 15 février — elle était connue à j, donc pas de leakage.


In [16]:
t0 = time.time()

def load_data(start=CONFIG['start_date']):
    # Yahoo Finance
    raw = yf.download(YF_TICKERS, start=start, auto_adjust=True, progress=False)['Close']
    raw.columns = [c.replace('^','IDX_').replace('-','_') for c in raw.columns]

    # Filtre couverture colonnes (pas lignes)
    coverage = raw.notna().mean()
    raw = raw.loc[:, coverage >= 0.90]
    raw = raw.ffill().dropna(how='all')

    # FRED
    fred_frames = []
    for name, series_id in FRED_SERIES.items():
        try:
            s = web.DataReader(series_id, 'fred', start).squeeze()
            s.name = f'FRED_{name}'
            fred_frames.append(s)
        except Exception as e:
            print(f"  [WARN] FRED {series_id}: {e}")

    if fred_frames:
        fred_df = pd.concat(fred_frames, axis=1).reindex(raw.index, method='ffill')
        raw = pd.concat([raw, fred_df], axis=1)

    print(f"  Dataset : {raw.shape[0]} jours × {raw.shape[1]} séries ({time.time()-t0:.1f}s)")
    return raw

df_raw = load_data()


  Dataset : 3783 jours × 64 séries (20.7s)


## 3. Features Time-Series : EGARCH, Kalman, HMM, Heston

### 3.1 EGARCH(1,1) — volatilité conditionnelle asymétrique

**Modèle** : l'EGARCH (Nelson, 1991) modélise la variance conditionnelle en log :
$$\log(\sigma^2_t) = \omega + \alpha(|z_{t-1}| - E|z|) + \gamma z_{t-1} + \beta \log(\sigma^2_{t-1})$$
Le terme $\gamma z_{t-1}$ capture l'**effet de levier** : $\gamma < 0$ signifie
qu'un choc négatif sur le SPX amplifie davantage la variance qu'un choc positif.
On utilise `dist='skewt'` (Student asymétrique) pour les queues épaisses du VIX.

**Feature extraite** : $\sigma^2_t$ (variance conditionnelle) et $\Delta\sigma^2_t$
(variation = signal directionnel, sans leakage car calculé à $t$ pour prédire $t+h$).

### 3.2 Filtre de Kalman — état latent du VIX

**Modèle état-espace** :
$$x_t = x_{t-1} + w_t \quad (\text{marche aléatoire latente})$$
$$y_t = x_t + v_t \quad (\text{observation bruitée})$$
Le filtre estime l'état caché $\hat{x}_t$ par l'algorithme de Kalman (optimal au sens des
moindres carrés si les bruits sont gaussiens). Les paramètres $Q$ (bruit d'état) et
$R$ (bruit d'observation) sont estimés par l'algorithme EM sur le **train uniquement**.

**Features extraites** :
- `kalman_residual = VIX_t − x̂_t` (sur-extension → signal mean-reversion)
- `kalman_innovation = VIX_t − x̂_{t|t-1}` (surprise → signal momentum) — **shiftée de 1j** pour éviter le leakage

### 3.3 HMM — régimes cachés probabilistes

**Modèle** : $P(\text{état}_{t+1} | \text{état}_t)$ — matrice de transition Markovienne.
Chaque état a sa propre distribution gaussienne. L'algorithme de Baum-Welch (EM)
estime les paramètres. On identifie l'état STRESS par la variance moyenne la plus haute.

**Feature extraite** : $P(\text{stress}_t)$ ∈ [0,1] — probabilité continue, plus riche qu'un état discret.

### 3.4 Proxies Heston — paramètres de volatilité stochastique

Le modèle de Heston (1993) suppose que la variance $v_t$ suit :
$$dv_t = \kappa(\theta - v_t)dt + \xi\sqrt{v_t}dW^v_t$$
On approxime les 5 paramètres depuis des données de marché (sans options) :
- $v_0 = (VIX/100)^2$, $\theta$ = RV rolling 60j, $\xi$ = VVIX/100, $\rho$ = corr rolling SPX-VIX, $\kappa$ = ln(2)/half-life AR(1)

**Espérances conditionnelles** (forme fermée) pour $h \in \{1,3,5,7\}$ :
$$E[v_{t+h}|v_t] = \theta + (v_0 - \theta)e^{-\kappa h}$$


In [74]:
t0 = time.time()
print("[Features TS] Calcul EGARCH, Kalman, HMM, Heston...")

def build_ts_features(df_raw, train_end_idx):
    """
    Construit toutes les features time-series.
    RÈGLE ABSOLUE : tout fit (EGARCH, Kalman, HMM) se fait sur df_raw.iloc[:train_end_idx].
    Le résultat est ensuite appliqué à tout le dataset, sans re-fit sur le test.
    Version corrigée : nettoyage strict des NaN/inf avant Kalman, HMM, Heston et VRP.
    """
    feats = pd.DataFrame(index=df_raw.index)

    # ── Colonnes principales ────────────────────────────────────────────────
    vix_col = 'IDX_VIX' if 'IDX_VIX' in df_raw.columns else [
        c for c in df_raw.columns if 'VIX' in c and 'VVIX' not in c
    ][0]

    spx_col = [c for c in df_raw.columns if 'GSPC' in c or 'SPY' in c][0]

    # Nettoyage robuste des séries de base
    vix = (
        df_raw[vix_col]
        .replace([np.inf, -np.inf], np.nan)
        .astype(float)
        .ffill()
        .bfill()
    )

    spx = (
        df_raw[spx_col]
        .replace([np.inf, -np.inf], np.nan)
        .astype(float)
        .ffill()
        .bfill()
    )

    # Sécurité si la première valeur reste problématique
    if not np.isfinite(vix.iloc[0]):
        vix.iloc[0] = vix.dropna().iloc[0]

    if not np.isfinite(spx.iloc[0]):
        spx.iloc[0] = spx.dropna().iloc[0]

    vix_ret = np.log(vix / vix.shift(1)).replace([np.inf, -np.inf], np.nan).fillna(0)
    spx_ret = np.log(spx / spx.shift(1)).replace([np.inf, -np.inf], np.nan).fillna(0)

    # ── EGARCH SPX ──────────────────────────────────────────────────────────
    try:
        spx_ret_train = (
            np.log(spx / spx.shift(1))
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
            .iloc[:train_end_idx]
            * 100
        )

        spx_ret_train = spx_ret_train.replace([np.inf, -np.inf], np.nan).fillna(0)

        am = arch_model(
            spx_ret_train,
            vol='EGARCH',
            p=1,
            q=1,
            dist='skewt',
            rescale=False
        )

        res_eg = am.fit(disp='off', show_warning=False)

        eg_full = res_eg.forecast(start=0, reindex=True)
        condvar = eg_full.variance.iloc[:, 0] / 10000
        condvar = condvar.reindex(df_raw.index, method='ffill')
        condvar = condvar.replace([np.inf, -np.inf], np.nan).ffill().bfill()

        feats['egarch_condvar'] = condvar
        feats['egarch_delta'] = condvar.diff()

        print(f"  EGARCH fit OK ({time.time()-t0:.1f}s)")

    except Exception as e:
        print(f"  [WARN] EGARCH: {e}")

    # ── Kalman corrigé ──────────────────────────────────────────────────────
    try:
        # Point clé : on crée une série VIX totalement finie.
        vix_clean = (
            vix
            .replace([np.inf, -np.inf], np.nan)
            .interpolate(method='linear')
            .ffill()
            .bfill()
            .astype(float)
        )

        # Sécurité supplémentaire
        if not np.isfinite(vix_clean.values).all():
            finite_mean = np.nanmean(vix_clean.replace([np.inf, -np.inf], np.nan).values)
            vix_clean = vix_clean.replace([np.inf, -np.inf], np.nan).fillna(finite_mean)

        train_vix_clean = vix_clean.iloc[:train_end_idx].values.reshape(-1, 1)
        full_vix_clean = vix_clean.values.reshape(-1, 1)

        # Vérification défensive avant pykalman
        if not np.isfinite(train_vix_clean).all():
            raise ValueError("train_vix_clean contient encore NaN/inf avant Kalman EM.")

        if not np.isfinite(full_vix_clean).all():
            raise ValueError("full_vix_clean contient encore NaN/inf avant Kalman filter.")

        kf = KalmanFilter(
            transition_matrices=np.array([[1.0]]),
            observation_matrices=np.array([[1.0]]),
            initial_state_mean=np.array([float(vix_clean.iloc[0])]),
            initial_state_covariance=np.array([[1.0]]),
            transition_covariance=np.array([[1e-3]]),
            observation_covariance=np.array([[1e-2]]),
            em_vars=['transition_covariance', 'observation_covariance']
        )

        kf = kf.em(train_vix_clean, n_iter=20)

        sm, _ = kf.filter(full_vix_clean)
        ss, _ = kf.smooth(full_vix_clean)

        kalman_filtered = pd.Series(sm[:, 0], index=df_raw.index)
        kalman_smooth = pd.Series(ss[:, 0], index=df_raw.index)

        feats['kalman_residual'] = (
            (vix_clean - kalman_filtered)
            .shift(1)
            .replace([np.inf, -np.inf], np.nan)
        )

        feats['kalman_innovation'] = (
            (vix_clean - kalman_smooth.shift(1))
            .shift(1)
            .replace([np.inf, -np.inf], np.nan)
        )

        print(f"  Kalman fit OK ({time.time()-t0:.1f}s)")

    except Exception as e:
        print(f"  [WARN] Kalman: {e}")
        feats['kalman_residual'] = np.nan
        feats['kalman_innovation'] = np.nan

    # ── HMM corrigé ─────────────────────────────────────────────────────────
    try:
        rv5 = vix_ret.pow(2).rolling(5, min_periods=3).mean()

        vix_train_mean = vix.iloc[:train_end_idx].mean()
        vix_train_std = vix.iloc[:train_end_idx].std()

        if not np.isfinite(vix_train_std) or vix_train_std <= 1e-12:
            vix_train_std = 1.0

        vix_n = (vix - vix_train_mean) / vix_train_std

        X_hmm = pd.DataFrame({
            'ret': vix_ret,
            'vol5': np.sqrt(rv5),
            'level': vix_n
        }, index=df_raw.index)

        X_hmm = X_hmm.replace([np.inf, -np.inf], np.nan).dropna()

        train_dates = df_raw.index[:train_end_idx]
        X_tr_df = X_hmm.loc[X_hmm.index.isin(train_dates)]

        if len(X_tr_df) < 50:
            raise ValueError("Pas assez d'observations propres pour entraîner le HMM.")

        X_tr = X_tr_df.values

        model_hmm = hmmlib.GaussianHMM(
            n_components=2,
            covariance_type='full',
            n_iter=200,
            random_state=SEED
        )

        model_hmm.fit(X_tr)

        states_tr = model_hmm.predict(X_tr)

        rv5_train = rv5.reindex(X_tr_df.index)
        state_vols = []

        for s in range(2):
            vals = rv5_train.values[states_tr == s]
            vals = vals[np.isfinite(vals)]
            state_vols.append(np.nanmean(vals) if len(vals) > 0 else -np.inf)

        stress_st = int(np.argmax(state_vols))

        proba_full = model_hmm.predict_proba(X_hmm.values)
        states_full = model_hmm.predict(X_hmm.values)

        feats['hmm_p_stress'] = (
            pd.Series(proba_full[:, stress_st], index=X_hmm.index)
            .reindex(df_raw.index)
            .replace([np.inf, -np.inf], np.nan)
        )

        feats['hmm_state'] = (
            pd.Series(states_full, index=X_hmm.index)
            .reindex(df_raw.index)
            .replace([np.inf, -np.inf], np.nan)
        )

        print(f"  HMM fit OK ({time.time()-t0:.1f}s)")

    except Exception as e:
        print(f"  [WARN] HMM: {e}")
        feats['hmm_p_stress'] = np.nan
        feats['hmm_state'] = np.nan

    # ── Heston proxies ──────────────────────────────────────────────────────
    v0 = (vix / 100).pow(2)
    theta = vix_ret.pow(2).rolling(60, min_periods=30).mean()

    vvix_col = [c for c in df_raw.columns if 'VVIX' in c]

    if vvix_col:
        xi = (
            df_raw[vvix_col[0]]
            .replace([np.inf, -np.inf], np.nan)
            .astype(float)
            .reindex(df_raw.index)
            .ffill()
            .bfill()
            / 100
        )
    else:
        xi = vix_ret.rolling(20, min_periods=10).std()

    rho = vix_ret.rolling(30, min_periods=15).corr(spx_ret)

    def rolling_kappa(series, w=252):
        kappa = pd.Series(np.nan, index=series.index)
        s = series.replace([np.inf, -np.inf], np.nan).ffill().bfill()

        for i in range(w, len(s)):
            y = s.iloc[i-w+1:i+1].values
            x = s.iloc[i-w:i].values

            try:
                if not np.isfinite(x).all() or not np.isfinite(y).all():
                    continue

                beta = np.corrcoef(x, y)[0, 1]

                if np.isfinite(beta) and 0 < abs(beta) < 0.9999:
                    hl = -np.log(2) / np.log(abs(beta))

                    if np.isfinite(hl) and hl > 0:
                        kappa.iloc[i] = np.log(2) / hl

            except Exception:
                pass

        return kappa.replace([np.inf, -np.inf], np.nan)

    kappa = rolling_kappa(vix)

    print(f"  Heston kappa OK ({time.time()-t0:.1f}s)")

    feats['heston_v0'] = v0
    feats['heston_theta'] = theta
    feats['heston_xi'] = xi
    feats['heston_rho'] = rho
    feats['heston_kappa'] = kappa
    feats['heston_feller'] = (2 * kappa * theta) / xi.pow(2).replace(0, np.nan)
    feats['heston_v0_minus_theta'] = v0 - theta

    for h in CONFIG['horizons']:
        ev = theta + (v0 - theta) * np.exp(-kappa * h)
        ev = ev.replace([np.inf, -np.inf], np.nan)

        feats[f'heston_ev_h{h}'] = ev
        feats[f'heston_spread_h{h}'] = v0 - ev
        feats[f'heston_vol_h{h}'] = np.sqrt(ev.clip(lower=0)) * 100

    # ── VRP : Variance Risk Premium ─────────────────────────────────────────
    try:
        rv1d = vix_ret.pow(2)
        rv5d = rv1d.rolling(5, min_periods=3).mean()
        rv22d = rv1d.rolling(22, min_periods=10).mean()

        import statsmodels.api as sm

        rv_target = rv1d.shift(-22).rolling(22, min_periods=11).mean()

        har_df = pd.DataFrame({
            'rv1': rv1d,
            'rv5': rv5d,
            'rv22': rv22d,
            'y': rv_target
        }, index=df_raw.index)

        har_df = har_df.replace([np.inf, -np.inf], np.nan).dropna()

        train_dates = df_raw.index[:train_end_idx]
        har_train = har_df.loc[har_df.index.isin(train_dates)]

        if len(har_train) < 50:
            raise ValueError("Pas assez d'observations propres pour HAR-RV.")

        Xh = sm.add_constant(har_train[['rv1', 'rv5', 'rv22']], has_constant='add')
        yh = har_train['y']

        har_model = sm.OLS(yh, Xh).fit()

        Xfull = sm.add_constant(har_df[['rv1', 'rv5', 'rv22']], has_constant='add')
        rv_pred = har_model.predict(Xfull).reindex(df_raw.index)

        feats['VRP'] = (vix / 100).pow(2) - rv_pred

        vrp_train = feats['VRP'].iloc[:train_end_idx]
        vrp_mu = vrp_train.mean()
        vrp_sd = vrp_train.std()

        if not np.isfinite(vrp_sd) or vrp_sd <= 1e-12:
            vrp_sd = 1.0

        feats['VRP_zscore'] = (feats['VRP'] - vrp_mu) / vrp_sd

    except Exception as e:
        print(f"  [WARN] VRP: {e}")
        feats['VRP'] = np.nan
        feats['VRP_zscore'] = np.nan

    # ── Jump Intensity ──────────────────────────────────────────────────────
    sigma_60 = vix_ret.rolling(60, min_periods=30).std()
    is_jump = (vix_ret.abs() > 3 * sigma_60).astype(float)

    feats['jump_intensity_20d'] = is_jump.rolling(20, min_periods=10).mean()
    feats['jump_intensity_60d'] = is_jump.rolling(60, min_periods=30).mean()

    # ── Hawkes Process ──────────────────────────────────────────────────────
    sigma_hw = vix_ret.rolling(30, min_periods=15).std()
    jump_times = vix_ret.index[vix_ret.abs() > 2 * sigma_hw]

    hawkes = pd.Series(0.0, index=vix_ret.index)

    for i, t in enumerate(vix_ret.index):
        past = jump_times[jump_times < t]

        if len(past) > 0:
            days_since = np.array([(t - tj).days for tj in past])
            hawkes.iloc[i] = 0.3 + 0.3 * np.sum(np.exp(-0.1 * days_since))
        else:
            hawkes.iloc[i] = 0.3

    feats['hawkes_intensity'] = hawkes

    mu_hw = hawkes.iloc[:train_end_idx].mean()
    sd_hw = hawkes.iloc[:train_end_idx].std()

    if not np.isfinite(sd_hw) or sd_hw <= 1e-12:
        sd_hw = 1.0

    feats['hawkes_zscore'] = (hawkes - mu_hw) / sd_hw

    print(f"  VRP + Jump + Hawkes OK ({time.time()-t0:.1f}s)")

    # Nettoyage final strict
    feats = feats.replace([np.inf, -np.inf], np.nan)

    return feats


[Features TS] Calcul EGARCH, Kalman, HMM, Heston...


## 4. Construction de la cible d'amplitude

**4 classes par quantiles conditionnels au régime** (calculés sur le train uniquement) :

| Classe | Condition |
|---|---|
| DOWN_FORT | $r < q_{25}^{\text{régime}}$ |
| DOWN_FAIBLE | $q_{25} \le r < 0$ |
| UP_FAIBLE | $0 \le r < q_{75}^{\text{régime}}$ |
| UP_FORT | $r \ge q_{75}^{\text{régime}}$ |

**Jours flat** ($|r| < 0.3\%$) supprimés — ils correspondent au bruit de microstructure
(pas de mouvement réel du VIX) et dégradent le signal pour toutes les classes.

**Quantiles conditionnels** : un mouvement "fort" en régime CALM ($VIX \approx 12$)
n'a pas la même amplitude qu'en régime STRESS ($VIX \approx 35$) — les seuils
s'adaptent à la distribution locale de chaque régime.


In [75]:
def build_amplitude_target(vix_series, horizon, train_end_idx, regime_series=None):
    """
    Cible : rendement VIX à horizon h = (VIX_{t+h}/VIX_t) - 1
    Classes : DOWN_FORT / DOWN_FAIBLE / UP_FAIBLE / UP_FORT
    Quantiles calculés sur le train uniquement par régime.
    """
    vix = vix_series.copy()
    ret = (vix.shift(-horizon) / vix) - 1

    # Supprimer les jours flat
    flat_mask = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat_mask]

    # Régimes (VIX q33/q67 sur train)
    vix_train = vix.iloc[:train_end_idx]
    calm_thr   = vix_train.quantile(0.33)
    stress_thr = vix_train.quantile(0.67)
    regime = pd.Series('NORMAL', index=ret.index)
    regime[vix.reindex(ret.index) < calm_thr]    = 'CALM'
    regime[vix.reindex(ret.index) >= stress_thr] = 'STRESS'

    # Quantiles par régime (train uniquement)
    ret_train = ret.iloc[:train_end_idx]
    thresholds = {}
    for reg in ['CALM','NORMAL','STRESS']:
        sub = ret_train[regime.iloc[:train_end_idx] == reg]
        thresholds[reg] = (sub.quantile(0.25), sub.quantile(0.75)) if len(sub)>=20 else (ret_train.quantile(0.25), ret_train.quantile(0.75))

    def classify(r, reg):
        q25, q75 = thresholds.get(reg, (0,0))
        if r < q25: return 0  # DOWN_FORT
        if r < 0:   return 1  # DOWN_FAIBLE
        if r < q75: return 2  # UP_FAIBLE
        return 3               # UP_FORT

    target = pd.Series(
        [classify(r, regime[i]) for i, r in ret.items()],
        index=ret.index, name='VIX_Amplitude_Class'
    )
    print(f"  [Target h={horizon}j] {len(target)} obs, dist: {target.value_counts().to_dict()}")
    return target, regime, ret, thresholds


## 5. Feature Engineering avancé

### Volatilités alternatives

**Parkinson (1980)** : utilise le range journalier High-Low pour estimer la volatilité.
Plus efficace que la volatilité réalisée sur clôtures car elle utilise toute l'information intraday :
$$\sigma_P = \sqrt{\frac{1}{4\ln 2}\left(\ln\frac{H}{L}\right)^2}$$

**Garman-Klass (1980)** : combine Open, High, Low, Close :
$$\sigma_{GK} = \sqrt{0.5(\ln H/L)^2 - (2\ln 2 - 1)(\ln C/O)^2}$$

**Momentum et RSI** : le RSI (Relative Strength Index) mesure la vitesse et
l'amplitude des mouvements de prix sur une fenêtre de $n$ jours.
Il est borné entre 0 et 100 et signale les sur-achats ($>70$) et sur-ventes ($<30$).

### Interactions de features (SHAP-guided)
Les interactions paires (ratio, différence, produit, z-score relatif, MA croisée)
multiplient le pouvoir expressif sans overfitting si elles sont sélectionnées par SHAP
après génération.


In [76]:
def build_advanced_features(df_raw, vix_series, spx_series, train_end_idx):
    feats = pd.DataFrame(index=df_raw.index)
    vix = vix_series.ffill()
    spx = spx_series.ffill()
    vix_ret = np.log(vix/vix.shift(1))
    spx_ret = np.log(spx/spx.shift(1))

    # Volatilités amplitude
    feats['vix_vol_of_vol_5d']   = vix_ret.rolling(5,  min_periods=3).std()
    feats['vix_vol_of_vol_10d']  = vix_ret.rolling(10, min_periods=5).std()
    feats['vix_momentum_2d']     = vix.pct_change(2)
    feats['vix_momentum_3d']     = vix.pct_change(3)
    feats['vix_acceleration']    = vix_ret - vix_ret.shift(3)
    feats['vix_erratic_ratio']   = vix_ret.abs().rolling(5,min_periods=3).max() / vix_ret.abs().rolling(5,min_periods=3).mean().replace(0,np.nan)
    feats['vix_vol_ratio_5_60']  = vix_ret.rolling(5,min_periods=3).std() / vix_ret.rolling(60,min_periods=30).std().replace(0,np.nan)

    # SPX amplitude
    feats['spx_vol_5d']          = spx_ret.rolling(5, min_periods=3).std()
    feats['spx_abs_ret_max_5d']  = spx_ret.abs().rolling(5, min_periods=3).max()
    feats['spx_momentum_3d']     = spx.pct_change(3)
    feats['vix_spx_corr_30d']    = vix_ret.rolling(30, min_periods=15).corr(spx_ret)

    # VIX vs MAs
    for w in [5, 10, 20]:
        ma = vix.rolling(w, min_periods=w//2).mean()
        feats[f'vix_vs_ma{w}'] = (vix - ma) / ma.replace(0,np.nan)
        feats[f'vix_zscore_{w}d'] = (vix - ma) / vix.rolling(w,min_periods=w//2).std().replace(0,np.nan)

    # Rendements multi-horizons
    for h in [1,2,3,5,10,20]:
        feats[f'vix_ret_{h}d'] = vix.pct_change(h)
        feats[f'spx_ret_{h}d'] = spx.pct_change(h)

    # RSI (14 jours)
    def rsi(series, n=14):
        delta = series.diff()
        gain  = delta.clip(lower=0).rolling(n).mean()
        loss  = (-delta.clip(upper=0)).rolling(n).mean()
        rs    = gain / loss.replace(0,np.nan)
        return 100 - 100/(1+rs)
    feats['vix_rsi_14']  = rsi(vix,  14)
    feats['spx_rsi_14']  = rsi(spx,  14)

    # Bollinger Width
    for w in [10,20]:
        ma  = vix.rolling(w).mean()
        std = vix.rolling(w).std()
        feats[f'boll_width_{w}d'] = (2*std) / ma.replace(0,np.nan)

    # MACD (12-26-9)
    ema12 = vix.ewm(span=12).mean()
    ema26 = vix.ewm(span=26).mean()
    macd  = ema12 - ema26
    signal = macd.ewm(span=9).mean()
    feats['vix_macd']          = macd
    feats['vix_macd_signal']   = signal
    feats['vix_macd_hist']     = macd - signal

    # Rolling skewness et kurtosis (erraticité)
    feats['vix_roll_skew_20d'] = vix_ret.rolling(20).skew()
    feats['vix_roll_kurt_20d'] = vix_ret.rolling(20).kurt()

    # Distance au max/min rolling
    for w in [20,60]:
        roll_max = vix.rolling(w).max()
        roll_min = vix.rolling(w).min()
        feats[f'vix_dist_max_{w}d'] = (vix - roll_max) / roll_max.replace(0,np.nan)
        feats[f'vix_dist_min_{w}d'] = (vix - roll_min) / roll_min.replace(0,np.nan)

    # SPX drawdown
    roll_high = spx.rolling(252, min_periods=126).max()
    feats['spx_drawdown_252d'] = (spx - roll_high) / roll_high.replace(0,np.nan)

    # Normalisation z-score sur train uniquement
    for col in feats.columns:
        mu = feats[col].iloc[:train_end_idx].mean()
        sd = feats[col].iloc[:train_end_idx].std()
        if sd > 1e-8:
            feats[f'{col}_z'] = (feats[col] - mu) / sd

    return feats.replace([np.inf,-np.inf], np.nan)


## 6. Interactions de features guidées par SHAP

**Principe de la sélection en 2 passes** :
1. SHAP pilote sur features de base → top-40 les plus informatives
2. Génération d'interactions sur les top-20 (190 paires × 6 opérations = 1140 colonnes)
3. Deuxième SHAP sur (top-40 base + 1140 interactions) → top-30 final

**Types d'interactions** :
- **Ratio** $A/B$ : capture les déséquilibres relatifs (ex: VIX_zscore/NFCI)
- **Différence** $A-B$ : spread entre deux signaux
- **Produit** $A \times B$ : interaction multiplicative (amplification)
- **Z-score relatif** $(A-B)/\sigma(A-B)$ : normalisation dynamique du spread
- **MA croisée** $MA_w(A)/MA_w(B)$ : tendance relative lissée
- **Momentum croisé** $\text{ret}_5(A) \times B$ : accélération d'un signal pondérée par un autre

**Pourquoi pas toutes les paires ?** Le nombre de features exploserait en $O(n^2)$.
On limite aux top-20 par SHAP pour maximiser la densité de signal par colonne.


In [77]:
def generate_interactions(df, base_features, top_n=20, rolling_w=20, eps=1e-8):
    feats = [f for f in base_features[:top_n] if f in df.columns]
    cols  = {}
    for i in range(len(feats)):
        for j in range(i+1, len(feats)):
            fi, fj = feats[i], feats[j]
            si, sj = df[fi], df[fj]
            sd = sj.where(sj.abs() >= eps, np.nan)
            cols[f'{fi}__div__{fj}']     = si / sd
            cols[f'{fi}__minus__{fj}']   = si - sj
            cols[f'{fi}__prod__{fj}']    = si * sj
            diff = si - sj
            rs   = diff.rolling(rolling_w, min_periods=rolling_w//2).std()
            cols[f'{fi}__zrel__{fj}']    = diff / rs.replace(0, np.nan)
            mai = si.rolling(rolling_w, min_periods=rolling_w//2).mean()
            maj = sj.rolling(rolling_w, min_periods=rolling_w//2).mean()
            cols[f'{fi}__macross__{fj}'] = mai / maj.where(maj.abs()>=eps, np.nan)
            cols[f'{fi}__ret5x__{fj}']   = si.pct_change(5) * sj
    idf = pd.DataFrame(cols, index=df.index).replace([np.inf,-np.inf], np.nan)
    idf = idf.dropna(axis=1, how='all')
    print(f"  [INTER] {len(feats)}f → {idf.shape[1]} interactions")
    return idf

def shap_select_features(X_train, y_train, top_n, label=''):
    try:
        from xgboost import XGBClassifier
        from sklearn.preprocessing import LabelEncoder

        # Nettoyage défensif
        X_clean = X_train.copy()
        if hasattr(X_clean, 'replace'):
            X_clean = X_clean.replace([np.inf, -np.inf], np.nan).fillna(0)
        else:
            X_clean = np.nan_to_num(X_clean, nan=0.0, posinf=0.0, neginf=0.0)

        # Supprimer colonnes encore problématiques
        if hasattr(X_clean, 'columns'):
            bad_cols = [c for c in X_clean.columns
                        if not np.isfinite(X_clean[c].values).all()]
            if bad_cols:
                X_clean = X_clean.drop(columns=bad_cols)

        le = LabelEncoder()
        y_enc = le.fit_transform(y_train)
        pilot = XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.05,
                               subsample=0.8, eval_metric='mlogloss',
                               objective='multi:softprob', random_state=SEED, n_jobs=-1)
        pilot.fit(X_clean.values if hasattr(X_clean,'values') else X_clean, y_enc)
        expl  = shap.TreeExplainer(pilot)
        sv    = expl.shap_values((X_clean.values if hasattr(X_clean,'values') else X_clean)[:500])
        if isinstance(sv, list):
            arr = np.mean([np.abs(s) for s in sv], axis=0)
        elif np.array(sv).ndim == 3:
            arr = np.abs(sv).mean(axis=2)
        else:
            arr = np.abs(sv)
        scores = pd.Series(arr.mean(axis=0), index=X_clean.columns if hasattr(X_clean,'columns') else range(arr.shape[1]))
        top = scores.sort_values(ascending=False).head(top_n).index.tolist()
        if label: print(f"  [SHAP {label}] top-{top_n}/{len(scores)}")
        return top, scores
    except ImportError:
        from sklearn.feature_selection import mutual_info_classif
        X_clean = np.nan_to_num(X_train.values if hasattr(X_train,'values') else X_train,
                                 nan=0.0, posinf=0.0, neginf=0.0)
        mi = mutual_info_classif(X_clean, y_train, random_state=SEED)
        cols = X_train.columns if hasattr(X_train,'columns') else range(len(mi))
        scores = pd.Series(mi, index=cols)
        return scores.nlargest(top_n).index.tolist(), scores

## 7. Dataset PyTorch — Fenêtres glissantes (lookback)

**Principe des séquences** : au lieu de donner un vecteur de features pour $t$,
on donne une matrice de shape $(lookback, n\_features)$ représentant les
$lookback$ derniers jours. Cela permet aux modèles séquentiels (LSTM, Transformer)
d'apprendre les dépendances temporelles.

**Normalisation** : le `StandardScaler` est **fitté sur le train uniquement**,
puis appliqué (transform) sur val et test. Fitter sur val/test constituerait
un leakage de normalisation.

**Gestion des classes** : on encode les 4 classes en entiers 0-3.
La BCE (Binary Cross-Entropy) est remplacée par la CrossEntropy multi-classes.


In [78]:
class VIXAmplitudeDataset(Dataset):
    """
    Dataset PyTorch pour la prédiction d'amplitude à 4 classes.
    Retourne des séquences de shape (lookback, n_features) et des labels entiers.
    """
    def __init__(self, data, feature_cols, target_col=TARGET_COL,
                 lookback=CONFIG['lookback'], scaler=None, label_encoder=None):
        self.lookback = lookback
        X = data[feature_cols].copy().fillna(0).astype(float)
        y = data[target_col].values.astype(int)

        self.scaler = scaler if scaler is not None else RobustScaler()
        self.X = self.scaler.fit_transform(X) if scaler is None else self.scaler.transform(X)
        self.y = y
        self.n_features = self.X.shape[1]

    def __len__(self):
        return max(0, len(self.X) - self.lookback)

    def __getitem__(self, idx):
        x_seq = torch.tensor(self.X[idx:idx+self.lookback], dtype=torch.float32)
        y_val = torch.tensor(self.y[idx+self.lookback], dtype=torch.long)
        return x_seq, y_val


## 8. Architectures Deep Learning

### 8.1 LSTM — Long Short-Term Memory

**Problème des RNN classiques** : le gradient disparaît (vanishing gradient) sur
les longues séquences, ce qui empêche d'apprendre les dépendances à long terme.

**Solution LSTM (Hochreiter & Schmidhuber, 1997)** : des "portes" (gates) contrôlent
le flux d'information via 3 vecteurs appris :
- **Forget gate** $f_t = \sigma(W_f[h_{t-1}, x_t] + b_f)$ : que retenir du passé
- **Input gate** $i_t = \sigma(W_i[h_{t-1}, x_t] + b_i)$ : quelle nouvelle info intégrer
- **Output gate** $o_t = \sigma(W_o[h_{t-1}, x_t] + b_o)$ : quoi exposer en sortie
- **Cell state** $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ : mémoire à long terme

### 8.2 TCN — Temporal Convolutional Network

**Convolutions dilatées** : un filtre de taille $k$ à dilation $d$ couvre une
fenêtre effective de $d(k-1)+1$ sans paramètres supplémentaires. En empilant
des dilations $d = 1, 2, 4, 8, \ldots$, le champ réceptif croît exponentiellement.

**Avantage vs LSTM** : parallélisable (pas de dépendance séquentielle au moment
du calcul), stable à entraîner, et empiriquement compétitif sur les séries temporelles.

### 8.3 Transformer — Attention Multi-têtes

**Mécanisme d'attention (Vaswani et al., 2017)** :
$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
Chaque token (jour) peut directement "regarder" n'importe quel autre token
de la séquence, sans contrainte de localité. Les têtes multiples capturent
des patterns à différentes échelles temporelles.

### 8.4 N-BEATS (Oreshkin et al., 2020) — *Nouveau*

Architecture pure feed-forward sans récurrence. Empile des blocs de résiduels :
chaque bloc prédit un **backcast** (reconstruction du passé) et un **forecast** (prédiction).
La somme des forecasts donne la prédiction finale. Interprétable : les blocs
peuvent être contraints à des bases de Fourier ou polynomiales.

### 8.5 Temporal Fusion Transformer (TFT) — *Nouveau*

Architecture state-of-the-art de Google (2020) pour les séries temporelles.
Combine LSTM (encoder temporel) + attention multi-têtes + mécanismes de gating :
- **Variable Selection Network** : pondère automatiquement les features
- **Gated Residual Networks** : connexions résiduelles avec contrôle de flux
- **Interprétabilité** : poids d'attention directement lisibles


In [79]:
# ═══════════════════════════════════════════════════════════════════════════
# ARCHITECTURES DEEP LEARNING
# ═══════════════════════════════════════════════════════════════════════════

class VIX_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                             dropout=dropout, bidirectional=False)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc   = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 32),         nn.GELU(),
            nn.Linear(32, n_classes)
        )
    def forward(self, x):
        out, (hn, _) = self.lstm(x)
        return self.fc(self.norm(out[:, -1, :]))


class TCNBlock(nn.Module):
    def __init__(self, n_in, n_out, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.conv  = nn.utils.weight_norm(
            nn.Conv1d(n_in, n_out, kernel_size, padding=pad, dilation=dilation))
        self.drop  = nn.Dropout(dropout)
        self.skip  = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else None
        self.act   = nn.GELU()
    def forward(self, x):
        out = self.act(self.drop(self.conv(x)[:, :, :-self.conv.padding[0]] if self.conv.padding[0] > 0 else self.conv(x)))
        res = x if self.skip is None else self.skip(x)
        return self.act(out + res)

class VIX_TCN(nn.Module):
    def __init__(self, input_dim, channels=[32,64,128], kernel_size=3, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        layers = []
        in_ch  = input_dim
        for i, ch in enumerate(channels):
            layers.append(TCNBlock(in_ch, ch, kernel_size, dilation=2**i, dropout=dropout))
            in_ch = ch
        self.net = nn.Sequential(*layers)
        self.fc  = nn.Sequential(
            nn.Linear(channels[-1], 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.net(x)
        return self.fc(x.mean(dim=2))  # Global Average Pooling


class VIX_Transformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=3,
                 dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.proj    = nn.Linear(input_dim, d_model)
        # Positional encoding appris
        self.pos_emb = nn.Embedding(CONFIG['lookback'], d_model)
        enc_layer    = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=d_model*4,
                                                   dropout=dropout, batch_first=True,
                                                   norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.fc      = nn.Sequential(
            nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        B, T, _ = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x    = self.proj(x) + self.pos_emb(pos)
        out  = self.encoder(x)
        return self.fc(out[:, -1, :])  # dernier token


class VIX_CNNLSTM(nn.Module):
    def __init__(self, input_dim, conv_filters=64, lstm_hidden=128,
                 kernel_size=3, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, conv_filters, kernel_size=kernel_size, padding='same'),
            nn.GELU(), nn.BatchNorm1d(conv_filters), nn.Dropout(dropout)
        )
        self.lstm = nn.LSTM(conv_filters, lstm_hidden, batch_first=True, num_layers=2,
                             dropout=dropout)
        self.fc   = nn.Sequential(
            nn.Linear(lstm_hidden, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        x = self.conv(x.transpose(1,2)).transpose(1,2)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class NBeatsBlock(nn.Module):
    """
    Bloc N-BEATS : réseau feed-forward qui prédit backcast (reconstruction)
    et forecast (prédiction) à partir d'une séquence.
    """
    def __init__(self, input_size, theta_size, hidden_size=256):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, theta_size)
        )
    def forward(self, x):
        return self.fc(x)

class VIX_NBeats(nn.Module):
    """
    N-BEATS simplifié pour la classification 4 classes.
    Chaque bloc raffine le résidu de la prédiction précédente (doubly residual stacking).
    """
    def __init__(self, input_dim, lookback, n_blocks=3, hidden_size=256,
                 dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.lookback = lookback
        self.blocks   = nn.ModuleList([
            NBeatsBlock(input_dim * lookback, hidden_size, hidden_size)
            for _ in range(n_blocks)
        ])
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        # x : (batch, lookback, features) → flatten
        residual = x.view(x.size(0), -1)
        out = None
        for block in self.blocks:
            theta = self.drop(block(residual))
            out   = theta if out is None else out + theta
        return self.fc(out)


class GatedLinearUnit(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc   = nn.Linear(in_dim, out_dim)
        self.gate = nn.Linear(in_dim, out_dim)
    def forward(self, x):
        return self.fc(x) * torch.sigmoid(self.gate(x))

class VIX_TFT(nn.Module):
    """
    Temporal Fusion Transformer simplifié.
    - Variable Selection Network : pondération des features par importance
    - LSTM encoder temporel
    - Multi-head attention inter-temporelle
    - Sortie : 4 classes amplitude
    """
    def __init__(self, input_dim, d_model=128, nhead=4, lstm_layers=2,
                 dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        # Variable Selection Network
        self.vsn = nn.Sequential(
            nn.Linear(input_dim, d_model), nn.GELU(),
            nn.Linear(d_model, input_dim), nn.Softmax(dim=-1)
        )
        self.input_proj = nn.Linear(input_dim, d_model)

        # Encoder LSTM
        self.lstm = nn.LSTM(d_model, d_model, num_layers=lstm_layers,
                             batch_first=True, dropout=dropout)
        self.lstm_norm = nn.LayerNorm(d_model)

        # Attention inter-temporelle
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                dim_feedforward=d_model*2,
                                                dropout=dropout, batch_first=True,
                                                norm_first=True)
        self.attention = nn.TransformerEncoder(enc_layer, num_layers=2)

        # GLU + Residual
        self.glu  = GatedLinearUnit(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

        self.fc = nn.Sequential(
            nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        # Variable selection
        weights = self.vsn(x.mean(dim=1, keepdim=True))  # (B,1,F)
        x = x * weights
        x = self.input_proj(x)

        # LSTM encoding
        lstm_out, _ = self.lstm(x)
        lstm_out    = self.lstm_norm(lstm_out + x)  # résidu

        # Attention
        attn_out = self.attention(lstm_out)

        # GLU + résidu final
        out = self.norm(self.glu(attn_out) + attn_out)

        return self.fc(out[:, -1, :])


print("6 architectures définies : LSTM, TCN, Transformer, CNN-LSTM, N-BEATS, TFT")


6 architectures définies : LSTM, TCN, Transformer, CNN-LSTM, N-BEATS, TFT


## 9. Entraînement — Focal Loss et Label Smoothing

### Focal Loss (Lin et al., 2017)

La **Cross-Entropy standard** traite également tous les exemples. Pour des classes
déséquilibrées (DOWN_FORT rare), les exemples faciles (UP_FAIBLE fréquent et bien
classé) dominent le gradient et le modèle "oublie" les classes difficiles.

La **Focal Loss** réduit la contribution des exemples faciles :
$$\text{FL}(p_t) = -\alpha_t (1-p_t)^\gamma \log(p_t)$$
- $(1-p_t)^\gamma$ : facteur de focalisation — si $p_t \to 1$ (exemple facile), ce terme $\to 0$
- $\alpha_t$ : pondération par fréquence de classe inverse
- $\gamma = 2$ est le réglage standard

### Label Smoothing

Remplace les cibles one-hot dures ($y=1$ ou $y=0$) par des cibles douces :
$$y_{\text{smooth}} = y(1-\epsilon) + \frac{\epsilon}{K}$$
avec $\epsilon = 0.1$ et $K=4$ classes. Réduit la confiance excessive du modèle
et améliore la généralisation (régularisation).

### Cosine Annealing + Warm Restarts

Le learning rate suit $\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max}-\eta_{\min})(1+\cos(\pi t/T_{max}))$.
Après chaque cycle, le LR est reset — les warm restarts aident à sortir des minima locaux.


In [80]:
class FocalLoss(nn.Module):
    """
    Focal Loss pour la classification multi-classes déséquilibrée.
    gamma=2, alpha inversement proportionnel à la fréquence de classe.
    """
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.gamma   = gamma
        self.alpha   = alpha  # tensor de poids par classe
        self.ls      = label_smoothing
        self.reduction = reduction
        self.n_classes = 4

    def forward(self, logits, targets):
        n_cls = logits.size(1)
        # Label smoothing
        one_hot = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1)
        smooth  = one_hot * (1 - self.ls) + self.ls / n_cls

        log_prob = torch.log_softmax(logits, dim=1)
        prob     = log_prob.exp()

        # Poids alpha
        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[targets]
        else:
            alpha_t = 1.0

        focal_weight = (1 - prob) ** self.gamma
        loss = -(alpha_t.unsqueeze(1) * focal_weight * smooth * log_prob).sum(dim=1)

        return loss.mean() if self.reduction == 'mean' else loss.sum()


def compute_class_weights(y_train, n_classes=4):
    """Inverse frequency weighting."""
    counts = np.bincount(y_train, minlength=n_classes)
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float32)


def train_model(model, train_loader, val_loader, class_weights=None,
                epochs=CONFIG['epochs'], lr=CONFIG['lr'],
                weight_decay=CONFIG['weight_decay'], label='Modèle', patience=10):
    criterion = FocalLoss(gamma=2.0, alpha=class_weights, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

    best_val_loss = float('inf')
    best_state    = None
    wait          = 0
    history       = {'train_loss':[], 'val_loss':[], 'val_f1':[]}

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        model.eval()
        val_loss = 0.0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                logits  = model(bx)
                val_loss += criterion(logits, by).item()
                val_preds.extend(logits.argmax(1).cpu().numpy())
                val_targets.extend(by.cpu().numpy())

        val_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)
        history['train_loss'].append(train_loss/len(train_loader))
        history['val_loss'].append(val_loss/len(val_loader))
        history['val_f1'].append(val_f1)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k:v.cpu().clone() for k,v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  [{label}] Early stop @ epoch {epoch+1} ({time.time()-t0:.1f}s)")
                break

        if (epoch+1) % 10 == 0:
            print(f"  [{label}] ep {epoch+1}/{epochs} | val_f1={val_f1:.4f} | {time.time()-t0:.1f}s")

    if best_state:
        model.load_state_dict(best_state)
    return history


def evaluate_model(model, loader, label=''):
    model.eval()
    all_preds, all_probs, all_targets = [], [], []
    with torch.no_grad():
        for bx, by in loader:
            logits = model(bx.to(device))
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = logits.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_probs.extend(probs)
            all_targets.extend(by.numpy())

    y_true = np.array(all_targets)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    # Métriques hiérarchiques (L1: direction, L2/L3: amplitude)
    dir_map = {0:'DOWN', 1:'DOWN', 2:'UP', 3:'UP'}
    yd_t = [dir_map[y] for y in y_true]
    yd_p = [dir_map[y] for y in y_pred]

    metrics = {
        'F1_4cls':    f1_score(y_true, y_pred, average='macro', zero_division=0),
        'Acc_4cls':   accuracy_score(y_true, y_pred),
        'Acc_dir':    accuracy_score(yd_t, yd_p),
        'F1_dir':     f1_score(yd_t, yd_p, average='macro', zero_division=0),
        'F1_UP':      f1_score(yd_t, yd_p, pos_label='UP',   average='binary', zero_division=0),
        'F1_DOWN':    f1_score(yd_t, yd_p, pos_label='DOWN', average='binary', zero_division=0),
    }
    # L2/L3 : amplitude
    up_idx  = [i for i,y in enumerate(y_true) if dir_map[y]=='UP']
    dn_idx  = [i for i,y in enumerate(y_true) if dir_map[y]=='DOWN']
    if up_idx:
        yt_up = ['FORT' if y_true[i]==3 else 'FAIBLE' for i in up_idx]
        yp_up = ['FORT' if y_pred[i]==3 else 'FAIBLE' for i in up_idx]
        metrics['F1_UP_FORT']   = f1_score(yt_up, yp_up, pos_label='FORT', average='binary', zero_division=0)
        metrics['Acc_UP_sub']   = accuracy_score(yt_up, yp_up)
    if dn_idx:
        yt_dn = ['FORT' if y_true[i]==0 else 'FAIBLE' for i in dn_idx]
        yp_dn = ['FORT' if y_pred[i]==0 else 'FAIBLE' for i in dn_idx]
        metrics['F1_DOWN_FORT'] = f1_score(yt_dn, yp_dn, pos_label='FORT', average='binary', zero_division=0)
        metrics['Acc_DOWN_sub'] = accuracy_score(yt_dn, yp_dn)

    if label:
        print(f"  [{label}] F1_dir={metrics['F1_dir']:.4f} F1_UP_FORT={metrics.get('F1_UP_FORT',0):.4f} F1_DOWN_FORT={metrics.get('F1_DOWN_FORT',0):.4f}")
    return metrics, y_pred, y_prob


## 10. SMOTE adaptatif + Pipeline principal

### BorderlineSMOTE

Le SMOTE standard génère des exemples synthétiques entre tout exemple minoritaire
et ses voisins. **BorderlineSMOTE** se concentre uniquement sur les exemples en
frontière de classe (ceux dont au moins la moitié des $k$ voisins sont d'une autre classe).

**Pourquoi ?** Les exemples facilement classés loin de la frontière n'ont pas besoin
d'être suréchantillonnés — ils n'apportent pas d'information sur la frontière de décision.
Générer des exemples synthétiques près de la frontière maximise l'apprentissage utile.

**Sélection automatique** : on compare SMOTE, BorderlineSMOTE et SMOTETomek sur
un modèle pilote rapide par (horizon, régime) et on retient le meilleur.


In [81]:
def select_best_sampler(X_train, y_train):
    """Compare SMOTE, BorderlineSMOTE, SMOTETomek sur un RandomForest pilote."""
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score, StratifiedKFold

    samplers = {
        'SMOTE':          SMOTE(random_state=SEED),
        'BorderlineSMOTE':BorderlineSMOTE(random_state=SEED, kind='borderline-1'),
        'SMOTETomek':     SMOTETomek(random_state=SEED),
    }
    pilot = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=SEED, n_jobs=-1)
    best_name, best_f1, best_sampler = 'SMOTE', -1, samplers['SMOTE']

    for name, samp in samplers.items():
        try:
            Xr, yr = samp.fit_resample(X_train, y_train)
            # Cross-val temporelle basique (3 splits séquentiels)
            n = len(Xr)
            scores = []
            for fold in range(3):
                t_end = n * (fold+2) // 4
                v_end = n * (fold+3) // 4
                pilot.fit(Xr[:t_end], yr[:t_end])
                preds = pilot.predict(Xr[t_end:v_end])
                scores.append(f1_score(yr[t_end:v_end], preds, average='macro', zero_division=0))
            f1 = np.mean(scores)
            if f1 > best_f1:
                best_f1, best_name, best_sampler = f1, name, samp
        except Exception:
            pass
    return best_name, best_sampler


def run_full_pipeline(df_raw, horizon=5):
    """
    Pipeline complet pour un horizon donné :
    1. Features TS (EGARCH, Kalman, HMM, Heston)
    2. Features avancées (vol, momentum, interactions)
    3. Cible amplitude 4 classes
    4. SHAP sélection 2 passes
    5. SMOTE adaptatif
    6. Entraînement 6 architectures
    7. Ensemble pondéré
    """
    t_total = time.time()
    print(f"\n{'='*60}")
    print(f"PIPELINE h={horizon}j")
    print(f"{'='*60}")

    # ── Split 80/20 ──────────────────────────────────────────────────────────
    all_dates  = df_raw.dropna(how='all').index.sort_values()
    split_idx  = int(len(all_dates) * 0.80)
    split_date = all_dates[split_idx]
    print(f"  Split 80/20 : train → {all_dates[split_idx-1].date()} | test → {split_date.date()}")

    # ── Features TS ──────────────────────────────────────────────────────────
    ts_feats = build_ts_features(df_raw, split_idx)

    # ── Features avancées ────────────────────────────────────────────────────
    vix_col = 'IDX_VIX' if 'IDX_VIX' in df_raw.columns else [c for c in df_raw.columns if 'VIX' in c and 'VVIX' not in c][0]
    spx_col = [c for c in df_raw.columns if 'GSPC' in c or 'SPY' in c][0]
    adv_feats = build_advanced_features(df_raw, df_raw[vix_col], df_raw[spx_col], split_idx)

    # ── Rendements multi-horizons sur tous les tickers ────────────────────────
    ret_feats = []
    for col in df_raw.columns:
        for w in [1,5,21]:
            s = df_raw[col].pct_change(w)
            s.name = f'{col}_ret{w}d'
            ret_feats.append(s)
    ret_df = pd.concat(ret_feats, axis=1)

    # ── Merge ─────────────────────────────────────────────────────────────────
    df_all = pd.concat([df_raw, ts_feats, adv_feats, ret_df], axis=1)
    df_all = df_all.replace([np.inf,-np.inf], np.nan)

    # ── Cible amplitude ───────────────────────────────────────────────────────
    target, regime, vix_ret, thresholds = build_amplitude_target(
        df_raw[vix_col], horizon, split_idx)
    df_all = df_all.reindex(target.index)
    df_all[TARGET_COL] = target

    # ── Sélection SHAP 2 passes ───────────────────────────────────────────────
    feature_cols = [c for c in df_all.columns if c != TARGET_COL and c in df_all.columns]
    df_train = df_all.loc[df_all.index < split_date].dropna(subset=[TARGET_COL])
    df_test  = df_all.loc[df_all.index >= split_date].dropna(subset=[TARGET_COL])

    X_tr_base = df_train[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
    y_tr      = df_train[TARGET_COL].values.astype(int)

    sc_base = RobustScaler()
    X_tr_sc = pd.DataFrame(sc_base.fit_transform(X_tr_base), columns=feature_cols, index=df_train.index)
#    DEBUG — à ajouter juste avant "print(f'  SHAP pass 1')"
    print(f"  X_tr_sc shape: {X_tr_sc.shape}")
    print(f"  NaN count: {X_tr_sc.isna().sum().sum()}")
    print(f"  Inf count: {np.isinf(X_tr_sc.values).sum()}")
    bad = X_tr_sc.columns[X_tr_sc.isna().any() | np.isinf(X_tr_sc.values).any(axis=0)].tolist()
    print(f"  Colonnes problématiques ({len(bad)}): {bad[:10]}")
    print(f"  SHAP pass 1 ({len(feature_cols)} features)...")
    top_base, scores_base = shap_select_features(X_tr_sc, y_tr, CONFIG['top_n_shap'], 'base')

    print(f"  Génération interactions (top-20)...")
    idf = generate_interactions(df_train[top_base], top_base, top_n=20)
    df_train_ext = pd.concat([df_train[top_base], idf], axis=1)
    df_test_ext  = pd.concat([df_test[top_base],
                               generate_interactions(df_test[top_base], top_base, top_n=20)], axis=1)

    ext_cols = df_train_ext.columns.tolist()
    sc_ext   = RobustScaler()
    X_tr_ext = pd.DataFrame(sc_ext.fit_transform(df_train_ext.fillna(0)),
                              columns=ext_cols, index=df_train.index)

    print(f"  SHAP pass 2 ({len(ext_cols)} features)...")
    top_final, scores_final = shap_select_features(X_tr_ext, y_tr, CONFIG['top_n_final'], 'final')
    print(f"  Features finales : {len(top_final)}")

    # ── SMOTE adaptatif ───────────────────────────────────────────────────────
    X_final_tr = X_tr_ext[top_final].fillna(0)
    X_final_te = pd.DataFrame(sc_ext.transform(df_test_ext.fillna(0)),
                               columns=ext_cols, index=df_test.index)[top_final].fillna(0)
    y_te        = df_test[TARGET_COL].values.astype(int)

    sampler_name, best_sampler = select_best_sampler(X_final_tr.values, y_tr)
    print(f"  Sampler sélectionné : {sampler_name}")
    X_res, y_res = best_sampler.fit_resample(X_final_tr.values, y_tr)

    # ── Datasets PyTorch ──────────────────────────────────────────────────────
    # On reconstruit le DataFrame rééchantillonné pour VIXAmplitudeDataset
    df_res = pd.DataFrame(X_res, columns=top_final)
    df_res[TARGET_COL] = y_res
    df_te_final = pd.DataFrame(X_final_te, columns=top_final)
    df_te_final[TARGET_COL] = y_te

    val_split = int(len(df_res) * 0.85)
    sc_seq    = RobustScaler().fit(df_res[top_final].iloc[:val_split])

    ds_train = VIXAmplitudeDataset(df_res.iloc[:val_split], top_final, scaler=sc_seq)
    ds_val   = VIXAmplitudeDataset(df_res.iloc[val_split:], top_final, scaler=sc_seq)
    ds_test  = VIXAmplitudeDataset(df_te_final, top_final, scaler=sc_seq)

    dl_train = DataLoader(ds_train, batch_size=CONFIG['batch_size'], shuffle=True)
    dl_val   = DataLoader(ds_val,   batch_size=256)
    dl_test  = DataLoader(ds_test,  batch_size=256)

    input_dim    = len(top_final)
    class_weights = compute_class_weights(y_res)

    # ── Entraînement 6 modèles ────────────────────────────────────────────────
    models_def = {
        'LSTM':        VIX_LSTM(input_dim),
        'TCN':         VIX_TCN(input_dim),
        'Transformer': VIX_Transformer(input_dim),
        'CNN-LSTM':    VIX_CNNLSTM(input_dim),
        'N-BEATS':     VIX_NBeats(input_dim, CONFIG['lookback']),
        'TFT':         VIX_TFT(input_dim),
    }
    results_all = {}
    trained_models = {}
    histories = {}

    for name, model in models_def.items():
        print(f"\n  ── {name} ──")
        model = model.to(device)
        hist  = train_model(model, dl_train, dl_val, class_weights=class_weights, label=name)
        met, preds, probs = evaluate_model(model, dl_test, label=name)
        results_all[name]  = met
        trained_models[name] = model
        histories[name]      = hist

    # ── Ensemble pondéré par F1_dir ───────────────────────────────────────────
    weights = np.array([results_all[n]['F1_dir'] for n in models_def])
    weights = (weights - weights.min() + 1e-6)
    weights = weights / weights.sum()

    all_probs_list = []
    for name in models_def:
        _, _, probs = evaluate_model(trained_models[name], dl_test)
        all_probs_list.append(probs)

    ensemble_probs = np.sum([p*w for p,w in zip(all_probs_list,weights)], axis=0)
    ensemble_preds = ensemble_probs.argmax(axis=1)

    # Métriques ensemble
    dir_map = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}
    yd_t = [dir_map[y] for y in y_te[:len(ensemble_preds)]]
    yd_p = [dir_map[y] for y in ensemble_preds]
    ens_f1dir = f1_score(yd_t, yd_p, average='macro', zero_division=0)

    print(f"\n  Ensemble F1_dir = {ens_f1dir:.4f} (poids: {dict(zip(models_def.keys(), weights.round(3)))})")

    print(f"\n  Pipeline h={horizon}j terminé ({time.time()-t_total:.1f}s)")
    return {'models': trained_models, 'results': results_all, 'ensemble_f1': ens_f1dir,
            'features': top_final, 'scaler': sc_seq, 'histories': histories}


## 11. Exécution — tous les horizons

On lance le pipeline complet pour $h \in \{1, 3, 5, 7\}$.
Chaque horizon est **totalement indépendant** : feature engineering,
sélection SHAP, SMOTE, et modèles sont tous séparés.


In [82]:
all_results = {}

for h in CONFIG['horizons']:
    all_results[h] = run_full_pipeline(df_raw, horizon=h)

# Résumé global
print("\n" + "="*60)
print("RÉSUMÉ — ENSEMBLE F1_dir par horizon")
print("="*60)
for h, res in all_results.items():
    ens = res['ensemble_f1']
    best_indiv = max(res['results'].items(), key=lambda x: x[1]['F1_dir'])
    print(f"  h={h}j | Ensemble F1_dir={ens:.4f} | Best individuel: {best_indiv[0]} ({best_indiv[1]['F1_dir']:.4f})")



PIPELINE h=1j
  Split 80/20 : train → 2023-08-17 | test → 2023-08-18
  EGARCH fit OK (0.7s)
  Kalman fit OK (35.9s)
  HMM fit OK (36.1s)
  Heston kappa OK (37.1s)
  VRP + Jump + Hawkes OK (40.5s)
  [Target h=1j] 3502 obs, dist: {1: 1045, 3: 877, 0: 861, 2: 719}
  X_tr_sc shape: (2803, 373)
  NaN count: 0
  Inf count: 0
  Colonnes problématiques (0): []
  SHAP pass 1 (373 features)...
  [SHAP base] top-40/373
  Génération interactions (top-20)...
  [INTER] 20f → 1102 interactions
  [INTER] 20f → 1102 interactions
  SHAP pass 2 (1142 features)...
  [SHAP final] top-30/1142
  Features finales : 30
  Sampler sélectionné : SMOTE

  ── LSTM ──
  [LSTM] ep 10/60 | val_f1=0.3005 | 31.8s
  [LSTM] ep 20/60 | val_f1=0.2659 | 60.6s
  [LSTM] Early stop @ epoch 21 (63.3s)
  [LSTM] F1_dir=0.5552 F1_UP_FORT=0.2918 F1_DOWN_FORT=0.3761

  ── TCN ──
  [TCN] ep 10/60 | val_f1=0.1591 | 13.6s
  [TCN] ep 20/60 | val_f1=0.1566 | 27.0s
  [TCN] Early stop @ epoch 23 (30.5s)
  [TCN] F1_dir=0.5192 F1_UP_FORT=0.1

## 12. Explicabilité SHAP — Transformer

**Pourquoi SHAP sur les modèles DL ?**

Pour les réseaux de neurones, on utilise `shap.DeepExplainer` (basé sur DeepLIFT)
ou `shap.GradientExplainer`. Ces méthodes approximent les SHAP values en propageant
l'impact de chaque feature à travers le réseau.

**SHAP Beeswarm** : chaque point = une observation. L'axe x = contribution SHAP.
La couleur = valeur de la feature (rouge = haute, bleu = basse).
Permet de voir si une feature a une relation monotone ou non-linéaire avec la cible.

**SHAP Waterfall** : pour une seule prédiction, montre comment chaque feature
"pousse" la prédiction vers le haut ou le bas depuis la valeur de base.


In [83]:
def plot_shap_dl(model, X_sample, feature_names, label=''):
    """SHAP GradientExplainer pour les modèles PyTorch."""
    model.eval()
    # GradientExplainer sur un sous-échantillon (trop coûteux sur tout le test)
    X_tensor = torch.tensor(X_sample[:100].astype(np.float32)).unsqueeze(1).repeat(1, CONFIG['lookback'], 1).to(device)
    background = X_tensor[:50]
    test_input  = X_tensor[50:]

    try:
        explainer   = shap.GradientExplainer(model, background)
        shap_values = explainer.shap_values(test_input)  # list de (n, lookback, F) par classe
        # Moyenne sur le lookback et les classes
        sv_mean = np.mean([np.abs(sv).mean(axis=1) for sv in shap_values], axis=0)
        sv_df   = pd.DataFrame(sv_mean, columns=feature_names)

        # Bar plot
        top_feats = sv_df.mean().nlargest(20).index.tolist()
        fig, ax = plt.subplots(figsize=(10, 6))
        sv_df[top_feats].mean().sort_values().plot(kind='barh', ax=ax, color='steelblue')
        ax.set_title(f'SHAP Feature Importance — {label}')
        ax.set_xlabel('Mean |SHAP value|')
        plt.tight_layout()
        plt.show()

        # Beeswarm (première classe)
        shap.summary_plot(shap_values[0].mean(axis=1), sv_df, plot_type='dot',
                          feature_names=feature_names, max_display=20, show=True)
    except Exception as e:
        print(f"  [WARN] SHAP DL: {e}")

# Exécuter SHAP sur le meilleur modèle GLOBAL h=5j
if 5 in all_results:
    res5  = all_results[5]
    best_name = max(res5['results'], key=lambda x: res5['results'][x]['F1_dir'])
    best_model = res5['models'][best_name]
    feats5 = res5['features']
    print(f"SHAP sur {best_name} h=5j")
    # Note : X_sample doit être reconstruit depuis df_test — placeholder
    print("  (plot disponible après reconstruction du test set)")


SHAP sur Transformer h=5j
  (plot disponible après reconstruction du test set)


## 13. Walk-Forward Validation institutionnelle

**Principe** : on simule le déploiement réel du modèle.
À chaque pas $t$, le modèle ne voit que les données disponibles avant $t$.
Le gap de 5 jours évite toute contamination entre train et test
(les données de fin de période train pourraient être révisées).

**Expanding window** : le train s'étend à chaque fold (toute l'histoire disponible),
ce qui est plus réaliste qu'une fenêtre glissante fixe pour ce type de données.

**Métriques agrégées** : F1_dir_mean ± std sur les folds —
un std élevé signale qu'un modèle a overfit une période spécifique.


In [84]:
def walk_forward_dl(df_raw, horizon=5, n_folds=4, gap_days=5, min_test_days=60):
    """Walk-forward expanding window pour les modèles DL."""
    results_wf = []
    all_dates  = df_raw.dropna(how='all').index.sort_values()
    total_n    = len(all_dates)
    initial_tr = int(total_n * 0.60)  # 60% minimum pour le train initial

    fold_size  = (total_n - initial_tr - gap_days) // n_folds
    if fold_size < min_test_days:
        print(f"[WARN] Trop peu de données pour {n_folds} folds. Réduit à {n_folds//2}.")
        n_folds   = n_folds // 2
        fold_size = (total_n - initial_tr - gap_days) // n_folds

    vix_col = 'IDX_VIX' if 'IDX_VIX' in df_raw.columns else [c for c in df_raw.columns if 'VIX' in c and 'VVIX' not in c][0]

    for fold in range(n_folds):
        train_end_idx  = initial_tr + fold * fold_size
        test_start_idx = train_end_idx + gap_days
        test_end_idx   = min(train_end_idx + (fold+1)*fold_size + gap_days, total_n)

        if test_end_idx - test_start_idx < min_test_days:
            continue

        train_dates = all_dates[:train_end_idx]
        test_dates  = all_dates[test_start_idx:test_end_idx]

        print(f"  Fold {fold+1} | Train→{train_dates[-1].date()} | Test: {test_dates[0].date()}→{test_dates[-1].date()} ({len(test_dates)} jours)")

        # Features et cible
        ts_feats  = build_ts_features(df_raw, train_end_idx)
        adv_feats = build_advanced_features(df_raw, df_raw[vix_col],
                                             df_raw[[c for c in df_raw.columns if 'GSPC' in c or 'SPY' in c][0]],
                                             train_end_idx)
        df_fold = pd.concat([df_raw, ts_feats, adv_feats], axis=1).replace([np.inf,-np.inf], np.nan)
        target, _, _, _ = build_amplitude_target(df_raw[vix_col], horizon, train_end_idx)
        df_fold = df_fold.reindex(target.index)
        df_fold[TARGET_COL] = target

        df_tr = df_fold.loc[df_fold.index.isin(train_dates)].dropna(subset=[TARGET_COL])
        df_te = df_fold.loc[df_fold.index.isin(test_dates)].dropna(subset=[TARGET_COL])
        if len(df_tr) < 100 or len(df_te) < 20: continue

        feat_cols = [c for c in df_fold.columns if c != TARGET_COL]
        sc = RobustScaler()
        X_tr = sc.fit_transform(df_tr[feat_cols].fillna(0))
        X_te = sc.transform(df_te[feat_cols].fillna(0))
        y_tr = df_tr[TARGET_COL].values.astype(int)
        y_te = df_te[TARGET_COL].values.astype(int)

        # SMOTE sur le train
        try:
            sm = BorderlineSMOTE(random_state=SEED)
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)
        except: pass

        df_tr_r = pd.DataFrame(X_tr, columns=feat_cols)
        df_tr_r[TARGET_COL] = y_tr
        df_te_r = pd.DataFrame(X_te, columns=feat_cols)
        df_te_r[TARGET_COL] = y_te

        cw = compute_class_weights(y_tr)
        n_feats = len(feat_cols)

        model = VIX_TFT(n_feats).to(device)  # TFT comme modèle principal WF
        ds_tr = VIXAmplitudeDataset(df_tr_r, feat_cols)
        ds_te = VIXAmplitudeDataset(df_te_r, feat_cols, scaler=ds_tr.scaler)
        dl_tr = DataLoader(ds_tr, batch_size=CONFIG['batch_size'], shuffle=True)
        dl_te = DataLoader(ds_te, batch_size=256)

        # Val = 15% du train
        ds_tr2 = VIXAmplitudeDataset(df_tr_r.iloc[:int(len(df_tr_r)*0.85)], feat_cols)
        ds_va  = VIXAmplitudeDataset(df_tr_r.iloc[int(len(df_tr_r)*0.85):], feat_cols, scaler=ds_tr2.scaler)
        dl_tr2 = DataLoader(ds_tr2, batch_size=CONFIG['batch_size'], shuffle=True)
        dl_va  = DataLoader(ds_va, batch_size=256)

        train_model(model, dl_tr2, dl_va, class_weights=cw, epochs=30, label=f'WF-fold{fold+1}', patience=7)
        met, _, _ = evaluate_model(model, dl_te, label=f'WF-fold{fold+1}')
        met['fold'] = fold+1
        met['test_start'] = test_dates[0]
        met['test_end']   = test_dates[-1]
        results_wf.append(met)

    df_wf = pd.DataFrame(results_wf)
    print(f"\n  Walk-Forward h={horizon}j — F1_dir: {df_wf['F1_dir'].mean():.4f} ± {df_wf['F1_dir'].std():.4f}")
    return df_wf

# Lancer sur h=5
wf_results = walk_forward_dl(df_raw, horizon=5, n_folds=4)


  Fold 1 | Train→2020-09-21 | Test: 2020-09-29→2022-03-10 (377 jours)
  EGARCH fit OK (2333.2s)
  Kalman fit OK (2352.7s)
  HMM fit OK (2352.8s)
  Heston kappa OK (2353.7s)
  VRP + Jump + Hawkes OK (2355.4s)
  [Target h=5j] 3713 obs, dist: {1: 1230, 3: 984, 0: 772, 2: 727}
  [WF-fold1] ep 10/30 | val_f1=0.3430 | 53.5s
  [WF-fold1] Early stop @ epoch 13 (69.3s)
  [WF-fold1] F1_dir=0.5866 F1_UP_FORT=0.3333 F1_DOWN_FORT=0.3304
  Fold 2 | Train→2022-03-03 | Test: 2022-03-11→2025-01-31 (754 jours)
  EGARCH fit OK (2425.4s)
  Kalman fit OK (2447.2s)
  HMM fit OK (2447.3s)
  Heston kappa OK (2448.2s)
  VRP + Jump + Hawkes OK (2449.9s)
  [Target h=5j] 3713 obs, dist: {1: 1159, 3: 921, 0: 843, 2: 790}
  [WF-fold2] ep 10/30 | val_f1=0.3396 | 60.1s
  [WF-fold2] Early stop @ epoch 18 (108.3s)
  [WF-fold2] F1_dir=0.5771 F1_UP_FORT=0.5069 F1_DOWN_FORT=0.2765
  Fold 3 | Train→2023-08-14 | Test: 2023-08-22→2026-07-16 (755 jours)
  EGARCH fit OK (2559.3s)
  Kalman fit OK (2583.9s)
  HMM fit OK (2584.0s

## 14. Rapport final et comparaison

Synthèse des performances de tous les modèles DL vs les benchmarks ML
(LogisticRegression, XGBoost, RandomForest) établis dans les runs précédents.


In [85]:
# Références ML (runs précédents)
ML_REFS = {
    'h=5j GLOBAL LogReg N=9 (ML)':       {'F1_dir':0.634, 'F1_UP_FORT':0.406, 'F1_DOWN_FORT':0.575},
    'h=5j GLOBAL RandomForest N=8 (ML)': {'F1_dir':0.620, 'F1_UP_FORT':0.412, 'F1_DOWN_FORT':0.462},
    'h=5j STRESS XGBoost N=8 (ML)':      {'F1_dir':0.587, 'F1_UP_FORT':0.377, 'F1_DOWN_FORT':0.518},
}

rows = []
for h, res in all_results.items():
    for model_name, met in res['results'].items():
        rows.append({
            'Horizon': h,
            'Modèle': model_name,
            'F1_dir':       round(met.get('F1_dir',0), 4),
            'F1_UP_FORT':   round(met.get('F1_UP_FORT',0), 4),
            'F1_DOWN_FORT': round(met.get('F1_DOWN_FORT',0), 4),
            'F1_4cls':      round(met.get('F1_4cls',0), 4),
            'Acc_dir':      round(met.get('Acc_dir',0), 4),
        })
    rows.append({
        'Horizon': h, 'Modèle': 'ENSEMBLE',
        'F1_dir': round(res['ensemble_f1'],4),
        'F1_UP_FORT': None, 'F1_DOWN_FORT': None, 'F1_4cls': None, 'Acc_dir': None
    })

df_report = pd.DataFrame(rows)

print("="*70)
print("RÉSULTATS DL — tous horizons et modèles")
print("="*70)
print(df_report.to_string(index=False))

print("\n" + "="*70)
print("BENCHMARKS ML (runs précédents — pour comparaison)")
print("="*70)
for k, v in ML_REFS.items():
    print(f"  {k}: F1_dir={v['F1_dir']} F1_UP_FORT={v['F1_UP_FORT']} F1_DOWN_FORT={v['F1_DOWN_FORT']}")

# Export Excel
try:
    with pd.ExcelWriter('vix_dl_report.xlsx', engine='xlsxwriter') as w:
        df_report.to_excel(w, sheet_name='DL_Results', index=False)
        if 'df_wf' in dir():
            df_wf.to_excel(w, sheet_name='WalkForward', index=False)
        pd.DataFrame(ML_REFS).T.to_excel(w, sheet_name='ML_Benchmarks')
    print("\n[SAVE] vix_dl_report.xlsx")
except Exception as e:
    print(f"[WARN] Export Excel: {e}")


RÉSULTATS DL — tous horizons et modèles
 Horizon      Modèle  F1_dir  F1_UP_FORT  F1_DOWN_FORT  F1_4cls  Acc_dir
       1        LSTM  0.5552      0.2918        0.3761   0.3136   0.5634
       1         TCN  0.5192      0.1319        0.2273   0.2513   0.5280
       1 Transformer  0.4888      0.0241        0.1529   0.2021   0.5236
       1    CNN-LSTM  0.4857      0.0588        0.4427   0.2661   0.5619
       1     N-BEATS  0.5347      0.2136        0.3017   0.2770   0.5546
       1         TFT  0.4969      0.0723        0.4308   0.2838   0.5664
       1    ENSEMBLE  0.4801         NaN           NaN      NaN      NaN
       3        LSTM  0.5680      0.5341        0.2857   0.3060   0.5790
       3         TCN  0.4352      0.0000        0.0874   0.1634   0.4853
       3 Transformer  0.5790      0.4016        0.2765   0.3306   0.5790
       3    CNN-LSTM  0.5437      0.4393        0.4502   0.3126   0.5469
       3     N-BEATS  0.5339      0.4027        0.5112   0.2798   0.5385
       3   

## 15. Conclusions et prochaines étapes

### Ce que ce notebook apporte vs la version initiale

| Feature | Version initiale | Cette version |
|---|---|---|
| Cible | Direction binaire | Amplitude 4 classes (DOWN_FORT/FAIBLE, UP_FAIBLE/FORT) |
| Features TS | Rendements basiques | EGARCH, Kalman, HMM, Heston proxies, VRP, Hawkes |
| Interactions | Ratio + produit (top-5) | 6 types × top-20 → SHAP 2 passes |
| Architectures | LSTM, TCN, Transformer, CNN-LSTM, TFT-basic | + **N-BEATS**, **TFT complet avec VSN** |
| Loss | BCE binaire | Focal Loss multi-classes + Label Smoothing |
| Optimiseur | Adam | AdamW + Cosine Annealing avec Warm Restarts |
| SMOTE | Standard | Adaptatif (SMOTE vs BorderlineSMOTE vs SMOTETomek) |
| Normalisation | StandardScaler | RobustScaler (robuste aux outliers) |
| Walk-forward | Gap fixe | Expanding window + gap de 5 jours |
| Régimes | GMM 3 composantes | GMM + HMM K=2 (probabilités continues) |

### Prochaines étapes suggérées

1. **Calibration des probabilités** : Platt scaling ou Temperature scaling pour
   que les softmax du modèle reflètent de vraies probabilités (utile pour le Kelly sizing)
2. **Heston calibration vraie** : implémenter la calibration sur options SPY (`yfinance.option_chain`)
3. **Stacking DL + ML** : utiliser les probabilités des 6 modèles DL comme features
   d'entrée d'un méta-modèle XGBoost/LogReg
4. **Attention visualization** : extraire les poids d'attention du TFT pour identifier
   quels jours passés le modèle utilise le plus
